In [9]:
import requests
import websocket
import json
import threading
import time
import atexit


def on_message(socket_url, result):
    message = json.loads(result)
    print(f"Received Message from {socket_url}:")
    print(message)

def create_socket(url, symbols, interval, stop_event):
    print(f"Starting thread for interval: {interval}")

    combined_streams = "/".join([f"{symbol.lower()}@kline_{interval}" for symbol in symbols])
    socket_url = f"wss://stream.binance.com:9443/stream?streams={combined_streams}"

    # Create a WebSocket object without using WebSocketApp
    ws = websocket.create_connection(socket_url)
    
    subscribe_message = {
        "method": "SUBSCRIBE",
        "params": [f"{symbol.lower()}@kline_{interval}" for symbol in symbols],
        "id": 1
    }
    ws.send(json.dumps(subscribe_message))

    # Main loop to handle WebSocket messages
    while not stop_event.is_set():
        try:
            # Receive WebSocket message with a timeout
            result = ws.recv()
            if result:
                on_message(socket_url, result)

        except websocket.WebSocketTimeoutException as e:
            # Timeout occurred, no data received, continue to check stop_event
            print(f"Timeout occurred: {e}")
            continue
        except Exception as e:
            print(f"WebSocket error: {e}")
            break

        # Check if stop_event is set
        if stop_event.is_set():
            print("Stop event detected, closing WebSocket")
            break

    # Close WebSocket connection
    ws.close()
    print(f"Stopping thread for interval: {interval}")

    
def start_websocket_threads(symbols, intervals, stop_events):
    threads = []
    for interval in intervals:
        stop_event = threading.Event()
        stop_events.append(stop_event)
        thread_name = f"BinanceWebSocketThread_{interval}"
        thread = threading.Thread(target=create_socket, args=("wss://stream.binance.com:9443/ws", symbols, interval, stop_event), name=thread_name)
        thread.custom_attribute = "just_doublechecking_binance_websocket"  # Custom attribute
        threads.append(thread)
        thread.start()
    return threads

def stop_websocket_threads(threads, stop_events):
    for stop_event in stop_events:
        stop_event.set()
    for thread in threads:
        thread.join(timeout=5)

def cleanup():
    print("Cleaning up and stopping threads...")
    
    # Signal all stop events for tracked threads
    for stop_event in stop_events:
        stop_event.set()

    # Enumerate all active threads
    active_threads = threading.enumerate()


    # Wait for all threads to complete
    for thread in active_threads:
        if thread.name.startswith("BinanceWebSocketThread_") or hasattr(thread, "just_doublechecking_binance_websocket"): # Avoid joining the current thread
            thread.join(timeout=5)
            if thread.is_alive():
                print(f"Warning: Thread {thread.name} did not terminate")
    
    print("Clean up done")
    
def fetch_top_symbols(n=10):
    url = "https://fapi.binance.com/fapi/v1/ticker/24hr"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        
        # Filter symbols that end with 'USDT' and sort by quote volume
        usdt_symbols = [item for item in data if item['symbol'].endswith('USDT')]

        # Sort symbols by quote volume
        sorted_symbols = sorted(usdt_symbols, key=lambda x: float(x['quoteVolume']), reverse=True)
        top_symbols = [item['symbol'] for item in sorted_symbols[:n]]
        return top_symbols
    else:
        print("Error fetching market data")
        return []


if __name__ == "__main__":
    try:
        # Create separate threads for different time intervals
        intervals = ["1m","5m","15m","1h","4h","1d"]
        update_interval = 7  # 1 hour in seconds
        threads = []
        stop_events = []

        atexit.register(cleanup)


        while True:
            top_10_symbols = fetch_top_symbols()
            if top_10_symbols:
                print("Top 10 symbols by daily volume:", top_10_symbols)

                threads = start_websocket_threads(top_10_symbols, intervals, stop_events)

                # Wait for the update interval
                time.sleep(update_interval)

                # Stop and join existing threads
                stop_websocket_threads(threads, stop_events)
                
                threads = []
                stop_events = []

                # Check active threads
                active_threads = threading.enumerate()
                print("Active threads:", active_threads)

            else:
                print("Failed to fetch top symbols")
                break
                
    except KeyboardInterrupt:
        print("Script interrupted by user")
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        cleanup()
        



Top 10 symbols by daily volume: ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'SUIUSDT', 'ORDIUSDT', 'MANTAUSDT', '1000SATSUSDT', 'ALTUSDT', 'TRBUSDT', 'UMAUSDT']
Starting thread for interval: 1m
Starting thread for interval: 5m
Starting thread for interval: 15m
Starting thread for interval: 1h
Starting thread for interval: 4h
Starting thread for interval: 1d
Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1d/ethusdt@kline_1d/solusdt@kline_1d/suiusdt@kline_1d/ordiusdt@kline_1d/mantausdt@kline_1d/1000satsusdt@kline_1d/altusdt@kline_1d/trbusdt@kline_1d/umausdt@kline_1d:Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1m/ethusdt@kline_1m/solusdt@kline_1m/suiusdt@kline_1m/ordiusdt@kline_1m/mantausdt@kline_1m/1000satsusdt@kline_1m/altusdt@kline_1m/trbusdt@kline_1m/umausdt@kline_1m:
{'stream': 'suiusdt@kline_1m', 'data': {'e': 'kline', 'E': 1706345435675, 's': 'SUIUSDT', 'k': {'t': 1706345400000, 'T': 1706345459999, 's': 'SUIUSDT', 'i': '1

Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1d/ethusdt@kline_1d/solusdt@kline_1d/suiusdt@kline_1d/ordiusdt@kline_1d/mantausdt@kline_1d/1000satsusdt@kline_1d/altusdt@kline_1d/trbusdt@kline_1d/umausdt@kline_1d:Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_15m/ethusdt@kline_15m/solusdt@kline_15m/suiusdt@kline_15m/ordiusdt@kline_15m/mantausdt@kline_15m/1000satsusdt@kline_15m/altusdt@kline_15m/trbusdt@kline_15m/umausdt@kline_15m:
{'stream': 'solusdt@kline_15m', 'data': {'e': 'kline', 'E': 1706345436676, 's': 'SOLUSDT', 'k': {'t': 1706345100000, 'T': 1706345999999, 's': 'SOLUSDT', 'i': '15m', 'f': 450855566, 'L': 450857331, 'o': '91.18000000', 'c': '91.36000000', 'h': '91.43000000', 'l': '91.13000000', 'v': '14773.67000000', 'n': 1766, 'x': False, 'q': '1349360.24410000', 'V': '9564.01000000', 'Q': '873545.45270000', 'B': '0'}}}
Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_4h/ethusdt@k

Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_5m/ethusdt@kline_5m/solusdt@kline_5m/suiusdt@kline_5m/ordiusdt@kline_5m/mantausdt@kline_5m/1000satsusdt@kline_5m/altusdt@kline_5m/trbusdt@kline_5m/umausdt@kline_5m:Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1h/ethusdt@kline_1h/solusdt@kline_1h/suiusdt@kline_1h/ordiusdt@kline_1h/mantausdt@kline_1h/1000satsusdt@kline_1h/altusdt@kline_1h/trbusdt@kline_1h/umausdt@kline_1h:
{'stream': 'btcusdt@kline_1h', 'data': {'e': 'kline', 'E': 1706345437469, 's': 'BTCUSDT', 'k': {'t': 1706342400000, 'T': 1706345999999, 's': 'BTCUSDT', 'i': '1h', 'f': 3391755157, 'L': 3391788605, 'o': '41710.15000000', 'c': '41689.18000000', 'h': '41719.85000000', 'l': '41540.52000000', 'v': '906.37153000', 'n': 33449, 'x': False, 'q': '37712950.70736610', 'V': '476.79708000', 'Q': '19840268.28378790', 'B': '0'}}}
Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1m/ethusd

Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1d/ethusdt@kline_1d/solusdt@kline_1d/suiusdt@kline_1d/ordiusdt@kline_1d/mantausdt@kline_1d/1000satsusdt@kline_1d/altusdt@kline_1d/trbusdt@kline_1d/umausdt@kline_1d:Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_15m/ethusdt@kline_15m/solusdt@kline_15m/suiusdt@kline_15m/ordiusdt@kline_15m/mantausdt@kline_15m/1000satsusdt@kline_15m/altusdt@kline_15m/trbusdt@kline_15m/umausdt@kline_15m:
{'stream': 'ordiusdt@kline_15m', 'data': {'e': 'kline', 'E': 1706345438675, 's': 'ORDIUSDT', 'k': {'t': 1706345100000, 'T': 1706345999999, 's': 'ORDIUSDT', 'i': '15m', 'f': 39353850, 'L': 39355154, 'o': '56.65600000', 'c': '56.69400000', 'h': '56.84900000', 'l': '56.62100000', 'v': '6003.59000000', 'n': 1305, 'x': False, 'q': '340468.52695000', 'V': '3749.65000000', 'Q': '212576.15979000', 'B': '0'}}}
Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_15m/ethusdt@k

Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_15m/ethusdt@kline_15m/solusdt@kline_15m/suiusdt@kline_15m/ordiusdt@kline_15m/mantausdt@kline_15m/1000satsusdt@kline_15m/altusdt@kline_15m/trbusdt@kline_15m/umausdt@kline_15m:Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1h/ethusdt@kline_1h/solusdt@kline_1h/suiusdt@kline_1h/ordiusdt@kline_1h/mantausdt@kline_1h/1000satsusdt@kline_1h/altusdt@kline_1h/trbusdt@kline_1h/umausdt@kline_1h:
{'stream': '1000satsusdt@kline_1h', 'data': {'e': 'kline', 'E': 1706345439229, 's': '1000SATSUSDT', 'k': {'t': 1706342400000, 'T': 1706345999999, 's': '1000SATSUSDT', 'i': '1h', 'f': 22928537, 'L': 22945798, 'o': '0.00049210', 'c': '0.00049130', 'h': '0.00049900', 'l': '0.00048820', 'v': '5792816730.00', 'n': 17262, 'x': False, 'q': '2855860.63303030', 'V': '3092933007.00', 'Q': '1526410.49531680', 'B': '0'}}}
Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_5m/e

...

Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1m/ethusdt@kline_1m/solusdt@kline_1m/suiusdt@kline_1m/ordiusdt@kline_1m/mantausdt@kline_1m/1000satsusdt@kline_1m/altusdt@kline_1m/trbusdt@kline_1m/umausdt@kline_1m:Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_5m/ethusdt@kline_5m/solusdt@kline_5m/suiusdt@kline_5m/ordiusdt@kline_5m/mantausdt@kline_5m/1000satsusdt@kline_5m/altusdt@kline_5m/trbusdt@kline_5m/umausdt@kline_5m:
{'stream': 'mantausdt@kline_5m', 'data': {'e': 'kline', 'E': 1706345448210, 's': 'MANTAUSDT', 'k': {'t': 1706345400000, 'T': 1706345699999, 's': 'MANTAUSDT', 'i': '5m', 'f': 10190066, 'L': 10190417, 'o': '3.43716000', 'c': '3.42743000', 'h': '3.43790000', 'l': '3.42606000', 'v': '21458.30000000', 'n': 352, 'x': False, 'q': '73652.44353100', 'V': '3426.40000000', 'Q': '11758.44329700', 'B': '0'}}}
Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_15m/ethusdt@kline_15m/solusd

Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1d/ethusdt@kline_1d/solusdt@kline_1d/suiusdt@kline_1d/ordiusdt@kline_1d/mantausdt@kline_1d/1000satsusdt@kline_1d/altusdt@kline_1d/trbusdt@kline_1d/umausdt@kline_1d:Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_5m/ethusdt@kline_5m/solusdt@kline_5m/suiusdt@kline_5m/ordiusdt@kline_5m/mantausdt@kline_5m/1000satsusdt@kline_5m/altusdt@kline_5m/trbusdt@kline_5m/umausdt@kline_5m:
{'stream': 'ordiusdt@kline_5m', 'data': {'e': 'kline', 'E': 1706345448670, 's': 'ORDIUSDT', 'k': {'t': 1706345400000, 'T': 1706345699999, 's': 'ORDIUSDT', 'i': '5m', 'f': 39355074, 'L': 39355202, 'o': '56.75700000', 'c': '56.74400000', 'h': '56.75700000', 'l': '56.68900000', 'v': '320.75000000', 'n': 129, 'x': False, 'q': '18190.46006000', 'V': '96.73000000', 'Q': '5486.38484000', 'B': '0'}}}

{'stream': 'ordiusdt@kline_1d', 'data': {'e': 'kline', 'E': 1706345448670, 's': 'ORDIUSDT', 'k': {'t': 170631360

Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_1h/ethusdt@kline_1h/solusdt@kline_1h/suiusdt@kline_1h/ordiusdt@kline_1h/mantausdt@kline_1h/1000satsusdt@kline_1h/altusdt@kline_1h/trbusdt@kline_1h/umausdt@kline_1h:Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_5m/ethusdt@kline_5m/solusdt@kline_5m/suiusdt@kline_5m/ordiusdt@kline_5m/mantausdt@kline_5m/1000satsusdt@kline_5m/altusdt@kline_5m/trbusdt@kline_5m/umausdt@kline_5m:
{'stream': 'suiusdt@kline_5m', 'data': {'e': 'kline', 'E': 1706345449676, 's': 'SUIUSDT', 'k': {'t': 1706345400000, 'T': 1706345699999, 's': 'SUIUSDT', 'i': '5m', 'f': 39657199, 'L': 39657517, 'o': '1.40000000', 'c': '1.40280000', 'h': '1.40320000', 'l': '1.39910000', 'v': '67744.20000000', 'n': 319, 'x': False, 'q': '94892.37477000', 'V': '45052.30000000', 'Q': '63113.15789000', 'B': '0'}}}
Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_5m/ethusdt@kline_5m/solusdt@kline

Stopping thread for interval: 1h
Active threads: [<_MainThread(MainThread, started 8226004480)>, <Thread(IOPub, started daemon 6154612736)>, <Heartbeat(Heartbeat, started daemon 6171439104)>, <Thread(Thread-3 (_watch_pipe_fd), started daemon 6189412352)>, <Thread(Thread-4 (_watch_pipe_fd), started daemon 6206238720)>, <ControlThread(Control, started daemon 6223065088)>, <HistorySavingThread(IPythonHistorySavingThread, started 6239891456)>, <ParentPollerUnix(Thread-2, started daemon 6257291264)>]
Top 10 symbols by daily volume: ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'SUIUSDT', 'ORDIUSDT', 'MANTAUSDT', '1000SATSUSDT', 'ALTUSDT', 'TRBUSDT', 'UMAUSDT']
Starting thread for interval: 1m
Starting thread for interval: 5m
Starting thread for interval: 15m
Starting thread for interval: 1h
Starting thread for interval: 4h
Starting thread for interval: 1d
Received Message from wss://stream.binance.com:9443/stream?streams=btcusdt@kline_15m/ethusdt@kline_15m/solusdt@kline_15m/suiusdt@kline_15m/ordiusdt@k

Stopping thread for interval: 15m
Stopping thread for interval: 4h
Stopping thread for interval: 1d
Stopping thread for interval: 1h
Stopping thread for interval: 1m
Stopping thread for interval: 5m
Clean up done


In [8]:
threading.enumerate()

[<_MainThread(MainThread, started 8226004480)>,
 <Thread(IOPub, started daemon 6154612736)>,
 <Heartbeat(Heartbeat, started daemon 6171439104)>,
 <Thread(Thread-3 (_watch_pipe_fd), started daemon 6189412352)>,
 <Thread(Thread-4 (_watch_pipe_fd), started daemon 6206238720)>,
 <ControlThread(Control, started daemon 6223065088)>,
 <HistorySavingThread(IPythonHistorySavingThread, started 6239891456)>,
 <ParentPollerUnix(Thread-2, started daemon 6257291264)>]

KeyboardInterrupt: 